# Creating metabolic models
This Jupyter Notebook uses carveme and cobrapy to create metabolic models of 7 bacterial species based on their protein fasta files and loads them into this notebook. Memote Reports are created to check model quality.

!! Ready to use models are provided (see GitHub), if you want to dive into our analyses, you can directly skip to Notebook 03

## Imports

In [1]:
import os
import subprocess
from cobra.io import read_sbml_model, write_sbml_model
import pandas as pd
import ast

# CarveMe with soft constraints
Soft constraints allow us to exclude reactions from the model; this is helpful for us to exclude duplicate metabolites/reactions further down the pipeline. Because of detailed manual curation efforts regarding charge balancing in a first attempt without soft constraints we identified model(s) that have reactions only coming from this model and creating duplicates in our models.

see also: https://carveme.readthedocs.io/en/latest/advanced.html#experimental-constraints

Currently, we are excluding iYO844 (Oh YK, Palsson BO, Park SM, Schilling CH, Mahadevan R. Genome-scale reconstruction of metabolic network in Bacillus subtilis based on high-throughput phenotyping and gene essentiality data. J Biol Chem. 2007;282(39):28791-28799. doi:10.1074/jbc.M703759200.)

In [46]:
# Through BIGGs API, data was collected showing which reaction is in which model
rxn_model_origin = pd.read_csv("../Datasets/BIGG/reactions_bigg_model_origin.csv", quotechar='"')
rxn_model_origin["bigg_models"] = rxn_model_origin["bigg_models"].apply(ast.literal_eval)

In [47]:
rxn_model_origin

,bigg_id,name,bigg_models
0,11DOCRTSTRNtm,11-deoxycorticosterone intracellular transport,"[Recon3D, iMM1415, iCHOv1, RECON1, iCHOv1_DG44]"
1,10FTHF5GLUtl,"5-glutamyl-10FTHF transport, lysosomal","[Recon3D, iMM1415, iCHOv1, RECON1, iCHOv1_DG44]"
2,11DOCRTSTRNtr,11-deoxycorticosterone intracellular transport,"[Recon3D, iMM1415, iCHOv1, RECON1, iCHOv1_DG44]"
3,10FTHFtm,10-Formyltetrahydrofolate mitochondrial transp...,"[Recon3D, iMM1415, iCHOv1, RECON1, iCHOv1_DG44]"
4,10FTHF7GLUtl,"7-glutamyl-10FTHF transport, lysosomal","[Recon3D, iMM1415, iCHOv1, iAT_PLT_636, RECON1..."
...,...,...,...
28297,ZYMSTESTH_SCe,Zymosterol ester hydrolase yeast specific ex...,[iMM904]
28298,ZYMSTR,Zymosterol reductase,[iRC1080]
28299,ZYMSTt,Zymosterol reversible transport,"[iMM904, iND750]"
28300,Zn2tex,Zinc (Zn+2) transport via diffusion (extracell...,"[iECO103_1326, iE2348C_1286, iEC1344_C, iEC134..."


In [10]:
filtered_false_reactions = pd.read_csv("../Models/01_carveme/exclude_reaction.tsv", quotechar='"', sep="\t", header=None)

In [11]:
filtered_false_reactions
# for all reactions a "R_" was added as a prefix in accordance with the format that carveme is expecting as an input for their soft constraints; the zero in second column indicates carveme that these reactions cannot be used in the model

,0,1
0,R_23CN2P1,0
1,R_2PGLYCt6,0
2,R_3AMBAt2,0
3,R_3AMPt6,0
4,R_3CMPt6,0
...,...,...
315,R_UNK5,0
316,R_UPP3MT_2,0
317,R_X00002,0
318,R_XYL14BASE,0


In [35]:
# Quick check that we excluded only iYO844 reactions
for value in filtered_false_reactions[0]:
    match = rxn_model_origin[rxn_model_origin["bigg_id"] == value[2:]]

    if not match.empty:
        print(match["bigg_models"].values[0])

['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']
['iYO844']

## CarveMe

In [2]:
# sequence_path is the location of the .faa files (protein fasta files)
# models_path is location where metabolic models should be saved
# constraints_path is location for the soft constraints file
sequence_path = "../Datasets"
models_path = "../Models/01_carveme"
constraints_path = "../Models/01_carveme/exclude_reaction.tsv"

Please make sure to have CarveMe version 1.6.2 installed! \
Carveme will create a list of warnings for each model because not all reactions that we specified for constraints are in each model.

Additionally cplex is necessary (with a license, community edition is not enough)

In [3]:
for root in os.listdir(sequence_path):
    if ".faa" in root:
        root_name = root[:3]
        print("creating model for: ", root_name)
        if "AA3" in root:
            subprocess.run(["carve", sequence_path+"/"+root, "-u", "grampos", "--solver", "cplex", "--soft", constraints_path, "--output", models_path+"/"+root_name+'.xml'])
        else:
            subprocess.run(["carve", sequence_path+"/"+root, "-u", "gramneg", "--solver", "cplex", "--soft", constraints_path, "--output", models_path+"/"+root_name+'.xml'])
print("done")

creating model for:  AA5


/home/schoenherrl/miniconda3/envs/carveme/lib/python3.10/site-packages/carveme/reconstruction/carving.py:220: UserWarning: Soft constraints contain reactions not in the model:
R_MOPSabc
R_EX_ala_L_gln__L_e
R_ALDD13
R_BLACT
R_LAAL2
R_LIPO1S24_BS
R_LAAL4
R_EX_ectoine_e
R_23CN2P1
R_LAAL34
R_LAAL18
R_LDH2
R_EX_ferxa_e
R_FACOAL140_ISO
R_LAAL30
R_PPTGS_BS
R_CDPDSP_BS
R_LAAL9
R_LAAL25
R_IPDPUPT
R_LAAL17
R_LAAL12
R_LAAL10
R_UGT2_BS
R_EX_lipt_e
R_LAAL16
R_ALDD10
R_PURabc1
R_PROLBabc
R_LIPO3S24_BS
R_EX_lanth_e
R_ARSNAt1
R_DKMD3M
R_LAAL41
R_EX_antim_e
R_HEMECS
R_LIPO2S24_BS
R_FACOAL171_anteiso
R_LAAL31
R_ALDD8
R_FACOAL150_anteiso
R_PHCYT_BS
R_FACOAL170_ISO
R_FACOAL170_anteiso
R_ALDD20x_1
R_CDGPT_BS
R_LAAL32
R_EX_amylase_e
R_EX_bilea_e
R_CYOR3m
R_ARSENBabc
R_ARSt1
R_FACOAL161_ISO
R_MLTG7
R_EX_prolb_e
R_TMPHi
R_BILEt4
R_LAAL28
R_GALS4
R_GBEZY
R_LAAL14
R_LAAL15
R_LAAL22
R_TECA3S45
R_LIPO4S24_BS
R_DHNAOT7
R_AMMQT7
R_DJENKabc
R_PUNP8
R_LAAL37
R_LAAL36
R_3AMBAt2
R_EX_djenk_e
R_EX_arsenb_e
R_LAAL33
R_DI

creating model for:  AA6


/home/schoenherrl/miniconda3/envs/carveme/lib/python3.10/site-packages/carveme/reconstruction/carving.py:220: UserWarning: Soft constraints contain reactions not in the model:
R_LAAL37
R_PHYT3
R_EX_djenk_e
R_ALDD14
R_GALS2
R_ARSENBabc
R_DHNAOT7
R_EX_lanth_e
R_PROLBabc
R_EX_bilea_e
R_LAAL22
R_XYL14BASE
R_LAAL19
R_UGT2_BS
R_PPTGS_BS
R_MLTG7
R_LAAL20
R_EX_arsenb_e
R_LAAL38
R_ARSNAt1
R_LIPO2S24_BS
R_TECA2S45
R_THIOGabc
R_DIPEPabc9
R_LAAL28
R_ALDD12
R_DKII
R_EX_mops_e
R_EX_L_alagly_e
R_LIPO1S24_BS
R_LAAL12
R_LAAL14
R_LAAL9
R_PYRIS
R_EX_amylase_e
R_GBEZY
R_ALDD20x_1
R_LAAL39
R_TECA3S45
R_TMPHi
R_CYOR3m
R_LAAL4
R_HEMEDS
R_HEMECS
R_HXAD
R_EX_antim_e
R_LAAL25
R_PURabc1
R_EX_ectoine_e
R_LAAL26
R_ALDD15
R_ALDD7
R_FACOAL160_ISO
R_EX_prolb_e
R_LDH2
R_LAAL6
R_EX_ala_L_his__L_e
R_LAAL8
R_TECA1S45
R_LIPO4S24_BS
R_CLPNS2_BS
R_AMMQT7
R_LAAL30
R_ARSt1
R_G3POA_BS
R_ECTOINEabc
R_LAAL21
R_UGT1_BS
R_EX_glycogen_e
R_LAAL32
R_EX_ala_L_leu__L_e
R_FACOAL171_ISO
R_CBL2abc
R_EX_lipt_e
R_AGPATr_BS
R_PGPPH_BS
R_LAAL

creating model for:  AA2


/home/schoenherrl/miniconda3/envs/carveme/lib/python3.10/site-packages/carveme/reconstruction/carving.py:220: UserWarning: Soft constraints contain reactions not in the model:
R_PGPPH_BS
R_EX_pur_e
R_LAAL25
R_SPMDAT3
R_IPDPUPT
R_FACOAL170_anteiso
R_FACOAL161_ISO
R_BILEt4
R_ALDD10
R_GALS2
R_PENCAC
R_LAAL9
R_LAAL18
R_DHNAOT7
R_DKMD3M
R_ALDD14
R_PYRIS
R_ALDD18
R_THIOGabc
R_G1Dx
R_LAAL5
R_CDGPT_BS
R_BIOMASS_BS_10
R_LDH2
R_EX_lipt_e
R_LAAL21
R_LAAL14
R_DIPEPabc7
R_LAAL38
R_LAAL4
R_ALDD8
R_23CN2P1
R_AGPATr_BS
R_LAAL33
R_GALS1
R_DIPEPabc4
R_LAAL23
R_EX_ferxa_e
R_DIPEPabc6
R_CYOO3
R_UGT_BS
R_ALDD15
R_EX_gly_gln__L_e
R_LAAL36
R_DJENKabc
R_EX_prolb_e
R_LAAL30
R_ANTIMt1
R_FACOAL150_ISO
R_ARSNAt1
R_HEMECS
R_LAAL27
R_LAAL39
R_FACOAL150_anteiso
R_ALDD7
R_DIPEPabc2
R_EX_glycogen_e
R_GBEZY
R_TECA1S45
R_MLTG7
R_LAAL28
R_HXAD
R_EX_ala_L_Thr__L_e
R_FERRICHabc
R_LAAL42
R_EX_tmp_e
R_LIPO1S24_BS
R_ALDD20x_1
R_FACOAL140_ISO
R_LAAL29
R_FERXAabc
R_PPTGS_BS
R_EX_L_alagly_e
R_EX_bilea_e
R_FACOAL160_ISO
R_EX_ala_

... (5 outputs left out)

/home/schoenherrl/miniconda3/envs/carveme/lib/python3.10/site-packages/carveme/reconstruction/carving.py:220: UserWarning: Soft constraints contain reactions not in the model:
R_LAAL10
R_LAAL4
R_GLYCOGENt
R_EX_glycogen_e
R_TEICH45
R_PGPPH_BS
R_LAAL31
R_EX_lipt_e
R_LAAL34
R_LAAL24
R_PHYT3
R_DIPEPabc7
R_ARSNAt1
R_ALDD18
R_PENCAC
R_LAAL40
R_THIOGabc
R_DIPEPabc4
R_BILEt4
R_EX_amylase_e
R_CYOO3
R_HEMEDS
R_FERXAabc
R_ALDD20x_1
R_G1Dx
R_DIPEPabc2
R_ALDD12
R_LAAL37
R_TMPHi
R_G1PPDM
R_ALDD15
R_EX_L_alagly_e
R_LAAL19
R_LAAL41
R_LAAL11
R_FACOAL171_anteiso
R_GBEZY
R_EX_mops_e
R_ALDD17
R_LAAL7
R_LAAL21
R_ALDD10
R_LAAL16
R_ARSENBabc
R_LAAL39
R_PSDC_BS
R_LAAL36
R_XYL14BASE
R_EX_ala_L_gln__L_e
R_LAAL22
R_ARSENBt2
R_DIPEPabc11
R_EX_arsni2_e
R_CLPNS2_BS
R_UGT1_BS
R_DHNAOT7
R_EX_thiog_e
R_EX_ala_L_Thr__L_e
R_LAAL2
R_HXAD
R_ANTIMt1
R_EX_djenk_e
R_AHSERL3
R_EX_gly_gln__L_e
R_LIPTabc
R_ALCD19_L
R_LAAL29
R_PROLBabc
R_GALS2
R_EX_ectoine_e
R_LAAL15
R_LAAL20
R_LAAL6
R_CDPDSP_BS
R_EX_antim_e
R_EX_arsna_e
R_LAAL3

creating model for:  AA1


/home/schoenherrl/miniconda3/envs/carveme/lib/python3.10/site-packages/carveme/reconstruction/carving.py:220: UserWarning: Soft constraints contain reactions not in the model:
R_LAAL30
R_ALDD10
R_DIPEPabc4
R_ALDD12
R_EX_amylase_e
R_DIPEPabc7
R_TMPt6
R_3AMBAt2
R_PHCYT_BS
R_ALDD15
R_LAAL15
R_EX_subtilisin_e
R_ALDD7
R_LAAL21
R_LAAL32
R_PYRIS
R_LAAL11
R_LAAL8
R_PUNP8
R_ALDD9
R_LAAL13
R_TECA2S45
R_TECA1S45
R_EX_ferrich_e
R_DKI
R_ARSt1
R_UGT_BS
R_FACOAL160_ISO
R_LAAL36
R_LAAL5
R_LDH2
R_HEMEDS
R_CDGPT_BS
R_FACOAL140_ISO
R_TECA3S45
R_DIPEPabc2
R_LAAL39
R_LAAL3
R_EX_lanth_e
R_EX_glycogen_e
R_PGPPH_BS
R_DIPEPabc11
R_EX_ala_L_Thr__L_e
R_EX_ferxa_e
R_ARSNAt1
R_LAAL19
R_EX_tmp_e
R_CYOO3
R_EX_ala_L_gln__L_e
R_HEMECS
R_LAAL31
R_LAAL35
R_GALS4
R_FACOAL150_anteiso
R_EX_arsna_e
R_SPMDAT3
R_G1PPDM
R_EX_ectoine_e
R_EX_gly_met__L_e
R_LAAL6
R_FACOAL170_anteiso
R_FACOAL171_anteiso
R_GALS1
R_EX_pur_e
R_TMPHi
R_ALDD14
R_CBL2abc
R_DIPEPabc5
R_UGT1_BS
R_LAAL10
R_ALCD19_L
R_AHSERL3
R_LYSLG_BS
R_LIPO4S24_BS
R_EX_b

done


## Import & save models
The xml files that carvme created are our sbml metabolic models; we use cplex as a solver; instead of the 3 letter abbreviation for each bacterium the models are named AA1 to AA7 based on the naming convention of Niu B et al., "Simplified and representative bacterial community of maize roots.", Proc Natl Acad Sci U S A, 2017 Mar 21;114(12):E2450-E2459

see also: https://www.ncbi.nlm.nih.gov/bioproject?term=PRJNA357031

AA1 - Sma \
AA2 - Bpi \
AA3 - Cpu \
AA4 - Elu \
AA5 - Cin \
AA6 - Hro \
AA7 - Ppu

In [4]:
# import models
models = {}
for model_name in (f for f in os.listdir(models_path) if f.endswith(".xml")):
    model = read_sbml_model(f"{models_path}/{model_name}")
    model.solver = "cplex"
    models[model_name[:3]] = model  # it takes first three letters of the xml filename as the model name

models = {key: models[key] for key in sorted(models.keys())}  # sorts the dictionary alphabetically (AA1...AA7)

# this is specifically for my 7 models to be able to access them more easily; all models are stored in the "model" dict and can be accessed e.g. through models["AA1"] depending what you set as model name
AA1, AA2, AA3, AA4, AA5, AA6, AA7 = [models[f"AA{i}"] for i in range(1, 8)]

Adding exchange reaction EX_25dkglcn_e with default bounds for boundary metabolite: 25dkglcn_e.
Adding exchange reaction EX_2ameph_e with default bounds for boundary metabolite: 2ameph_e.
Adding exchange reaction EX_2m35mdntha_e with default bounds for boundary metabolite: 2m35mdntha_e.
Adding exchange reaction EX_2obut_e with default bounds for boundary metabolite: 2obut_e.
Adding exchange reaction EX_34dhbz_e with default bounds for boundary metabolite: 34dhbz_e.
Adding exchange reaction EX_35dnta_e with default bounds for boundary metabolite: 35dnta_e.
Adding exchange reaction EX_3h4atb_e with default bounds for boundary metabolite: 3h4atb_e.
Adding exchange reaction EX_3hoxpac_e with default bounds for boundary metabolite: 3hoxpac_e.
Adding exchange reaction EX_3mb_e with default bounds for boundary metabolite: 3mb_e.
Adding exchange reaction EX_3oxoadp_e with default bounds for boundary metabolite: 3oxoadp_e.
Adding exchange reaction EX_4hba_e with default bounds for boundary meta

In [5]:
# save models as xml; no changes were introduced to the models yet; this is just testing that everything works correctly
for model_name, model in models.items():
    path = f"{models_path}/{model_name}.xml"
    write_sbml_model(model, path)

After model creation, the quality can be checked with MEMOTE 0.17.0 which was done on the command line. You can use the following command: \
memote report snapshot --filename "report_model.html" model.xml --solver cplex